In [1]:
import pandas as pd
import sqlite3

# Load and Explore the Data

In [2]:
# Connecting to the IPL database
conn = sqlite3.connect('database.sqlite')

# Query to obtain the structure of the database (list of tables)
query = '''

SELECT name
FROM sqlite_master
WHERE type='table'

'''

# Display of tables available in the database (global structure)
df_tables = pd.read_sql_query(query, conn)
df_tables

,name
0,Player
1,Extra_Runs
2,Batsman_Scored
3,Batting_Style
4,Bowling_Style
5,Country
6,Season
7,City
8,Outcome
9,Win_By


In [3]:
# Load all the tables and print their column names to identify common columns.
table_names = df_tables['name'].tolist()

for table in table_names:
    print(f"\n Table: {table}")
    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    print("Colums:", df.columns.tolist())



 Table: Player
Colums: ['Player_Id', 'Player_Name', 'DOB', 'Batting_hand', 'Bowling_skill', 'Country_Name']

 Table: Extra_Runs
Colums: ['Match_Id', 'Over_Id', 'Ball_Id', 'Extra_Type_Id', 'Extra_Runs', 'Innings_No']

 Table: Batsman_Scored
Colums: ['Match_Id', 'Over_Id', 'Ball_Id', 'Runs_Scored', 'Innings_No']

 Table: Batting_Style
Colums: ['Batting_Id', 'Batting_hand']

 Table: Bowling_Style
Colums: ['Bowling_Id', 'Bowling_skill']

 Table: Country
Colums: ['Country_Id', 'Country_Name']

 Table: Season
Colums: ['Season_Id', 'Man_of_the_Series', 'Orange_Cap', 'Purple_Cap', 'Season_Year']

 Table: City
Colums: ['City_Id', 'City_Name', 'Country_id']

 Table: Outcome
Colums: ['Outcome_Id', 'Outcome_Type']

 Table: Win_By
Colums: ['Win_Id', 'Win_Type']

 Table: Wicket_Taken
Colums: ['Match_Id', 'Over_Id', 'Ball_Id', 'Player_Out', 'Kind_Out', 'Fielders', 'Innings_No']

 Table: Venue
Colums: ['Venue_Id', 'Venue_Name', 'City_Id']

 Table: Extra_Type
Colums: ['Extra_Id', 'Extra_Name']

 Table

# Queries

**Query 1:** Select All Columns from Player’s Table

In [4]:
# Write and execute a SQL query to select all columns from the Player_Match table.

query = '''

SELECT *
FROM Player_Match

'''

df_query = pd.read_sql_query(query, conn)
df_query.columns

Index(['Match_Id', 'Player_Id', 'Role_Id', 'Team_Id'], dtype='object')

**Query 2:** Batsman vs Runs

In [5]:
# Write and execute a SQL query to calculate the total runs scored by each batsman.

query = '''

SELECT p.Player_Name, SUM(bs.Runs_Scored) AS total_runs
FROM Batsman_Scored bs
JOIN Player_Match pm ON pm.Match_Id = bs.Match_Id
JOIN Player p ON p.Player_Id = pm.Player_Id
GROUP BY p.Player_Name
ORDER BY total_runs DESC

'''

df_query = pd.read_sql_query(query, conn)
df_query

,Player_Name,total_runs
0,SK Raina,43101
1,MS Dhoni,41667
2,RG Sharma,41431
3,V Kohli,40563
4,KD Karthik,39424
...,...,...
464,NJ Rimmington,211
465,P Prasanth,183
466,DR Martyn,171
467,RV Pawar,117


**Query 3:** Fifties and Hundreds

In [6]:
# Write and execute a SQL query to calculate the number of fifties and hundreds scored by each batsman.

query = '''

WITH per_match_runs AS (
  SELECT
    bs.Match_Id,
    bs.Innings_No,
    b.Striker AS Player_Id,
    SUM(bs.Runs_Scored) AS total_runs
  FROM Batsman_Scored bs
  JOIN Ball_by_Ball b
    ON bs.Match_Id = b.Match_Id
   AND bs.Over_Id = b.Over_Id
   AND bs.Ball_Id = b.Ball_Id
   AND bs.Innings_No = b.Innings_No
  GROUP BY bs.Match_Id, bs.Innings_No, b.Striker
)

SELECT
  p.Player_Name,
  SUM(CASE WHEN pmr.total_runs BETWEEN 50 AND 99 THEN 1 ELSE 0 END) AS fifties,
  SUM(CASE WHEN pmr.total_runs >= 100 THEN 1 ELSE 0 END) AS hundreds
FROM per_match_runs pmr
JOIN Player p ON p.Player_Id = pmr.Player_Id
GROUP BY p.Player_Name
ORDER BY hundreds DESC, fifties DESC;


'''

df_query = pd.read_sql_query(query, conn)
df_query

,Player_Name,fifties,hundreds
0,CH Gayle,20,5
1,V Kohli,26,4
2,AB de Villiers,21,3
3,DA Warner,32,2
4,V Sehwag,16,2
...,...,...,...
429,Yashpal Singh,0,0
430,Younis Khan,0,0
431,YS Chahal,0,0
432,YV Takawale,0,0


**Query 4:** Best Bowling Figures

In [7]:
# Write and execute a SQL query to find the best bowling figures for each bowler.

query = '''

WITH bowler_performance AS (
  SELECT
    b.Bowler AS Player_Id,
    b.Match_Id,
    SUM(CASE WHEN wt.Player_Out IS NOT NULL THEN 1 ELSE 0 END) AS wickets_taken
  FROM Ball_by_Ball b
  LEFT JOIN Wicket_Taken wt
    ON b.Match_Id = wt.Match_Id
   AND b.Over_Id = wt.Over_Id
   AND b.Ball_Id = wt.Ball_Id
   AND b.Innings_No = wt.Innings_No
  GROUP BY b.Bowler, b.Match_Id
)

SELECT
  p.Player_Name,
  MAX(bp.wickets_taken) AS best_wickets_in_match
FROM bowler_performance bp
JOIN Player p ON p.Player_Id = bp.Player_Id
GROUP BY p.Player_Name
ORDER BY best_wickets_in_match DESC;



'''

df_query = pd.read_sql_query(query, conn)
df_query

,Player_Name,best_wickets_in_match
0,Sohail Tanvir,6
1,DJG Sammy,6
2,AD Russell,6
3,A Zampa,6
4,VY Mahesh,5
...,...,...
326,BJ Rohrer,0
327,B Chipli,0
328,AUK Pathan,0
329,AS Raut,0


**Query 5:** Comprehensive Career Metrics

In [8]:
# Combine all the previous chunks into a single comprehensive query to get detailed career metrics for players.

query = '''

WITH
runs_per_match AS (
    SELECT
        b.Match_Id,
        b.Innings_No,
        b.Striker AS Player_Id,
        SUM(bs.Runs_Scored) AS runs
    FROM Batsman_Scored bs
    JOIN Ball_by_Ball b ON bs.Match_Id = b.Match_Id
                       AND bs.Over_Id = b.Over_Id
                       AND bs.Ball_Id = b.Ball_Id
                       AND bs.Innings_No = b.Innings_No
    GROUP BY b.Match_Id, b.Innings_No, b.Striker
),

fifties_hundreds AS (
    SELECT
        Player_Id,
        SUM(CASE WHEN runs BETWEEN 50 AND 99 THEN 1 ELSE 0 END) AS fifties,
        SUM(CASE WHEN runs >= 100 THEN 1 ELSE 0 END) AS hundreds,
        SUM(runs) AS total_runs
    FROM runs_per_match
    GROUP BY Player_Id
),

balls_faced AS (
    SELECT
        Striker AS Player_Id,
        COUNT(*) AS balls_faced
    FROM Ball_by_Ball
    GROUP BY Striker
),

matches_played AS (
    SELECT
        Player_Id,
        COUNT(DISTINCT Match_Id) AS matches_played
    FROM Player_Match
    GROUP BY Player_Id
),

wickets_per_match AS (
    SELECT
        b.Bowler AS Player_Id,
        b.Match_Id,
        COUNT(wt.Player_Out) AS wickets
    FROM Ball_by_Ball b
    LEFT JOIN Wicket_Taken wt ON b.Match_Id = wt.Match_Id
                             AND b.Over_Id = wt.Over_Id
                             AND b.Ball_Id = wt.Ball_Id
                             AND b.Innings_No = wt.Innings_No
    GROUP BY b.Bowler, b.Match_Id
),

wickets_total AS (
    SELECT
        Player_Id,
        SUM(wickets) AS total_wickets,
        MAX(wickets) AS best_bowling
    FROM wickets_per_match
    GROUP BY Player_Id
)

SELECT
    p.Player_Name,
    COALESCE(fh.total_runs, 0) AS total_runs,
    COALESCE(fh.fifties, 0) AS fifties,
    COALESCE(fh.hundreds, 0) AS hundreds,
    COALESCE(mp.matches_played, 0) AS matches_played,
    COALESCE(bf.balls_faced, 0) AS balls_faced,
    COALESCE(wt.total_wickets, 0) AS wickets_taken,
    COALESCE(wt.best_bowling, 0) AS best_bowling
FROM Player p
LEFT JOIN fifties_hundreds fh ON p.Player_Id = fh.Player_Id
LEFT JOIN matches_played mp ON p.Player_Id = mp.Player_Id
LEFT JOIN balls_faced bf ON p.Player_Id = bf.Player_Id
LEFT JOIN wickets_total wt ON p.Player_Id = wt.Player_Id
ORDER BY total_runs DESC, wickets_taken DESC


'''

df_query = pd.read_sql_query(query, conn)
df_query

,Player_Name,total_runs,fifties,hundreds,matches_played,balls_faced,wickets_taken,best_bowling
0,SK Raina,4106,28,1,146,3059,29,2
1,V Kohli,4105,26,4,138,3237,5,2
2,RG Sharma,3874,29,1,142,2996,16,4
3,G Gambhir,3634,31,0,132,3028,0,0
4,CH Gayle,3447,20,5,91,2359,19,3
...,...,...,...,...,...,...,...,...
464,DL Chahar,0,0,0,2,0,0,0
465,P Dharmani,0,0,0,1,0,0,0
466,RV Pawar,0,0,0,1,0,0,0
467,KH Devdhar,0,0,0,1,0,0,0
